# Gold

Este notebook corresponde à **camada Gold** do pipeline de dados, sendo responsável pela padronização dos dados provenientes da camada Silver, disponíveis na tabela **`workspace.silver.tb_Sinistros_Transito_open_data`** no Databricks.

A camada Gold da arquitetura medallion é a etapa da pipeline de dados responsável pela disponibilização de dados refinados, agregados e prontos para consumo analítico. Nessa camada, são aplicadas regras de negócio, métricas e transformações que permitem gerar insights estratégicos, dashboards e análises preditivas.

Como o projeto utiliza dados de sinistros de trânsito da cidade do Recife, o foco da camada Gold foi a construção de tabelas analíticas capazes de responder questões relevantes para a mobilidade urbana e segurança no trânsito. Entre os exemplos de análises desenvolvidas estão: bairros com maior número de acidentes, horários de maior ocorrência, tipos de veículos mais envolvidos em sinistros e meios de transporte associados a acidentes fatais.



In [0]:
from pyspark.sql import functions as F
import pandas as pd

## Criação do Pandas Dataframe

In [0]:
df_pandas = spark.read.format("delta").load(
    "/Volumes/workspace/silver/dbs/Sinistros_Transito/Sinistros_Transito_open_data"
).toPandas()

In [0]:
df_pandas.head(100)

## Informações por bairro

### Queremos saber informações apuradas para cada bairro. Perguntas como: "Quais são os bairros com maior número de acidentes?", "Qual bairro tem acidentes mais fatais?" e várias outras podem ser respondidas a partir dessa tabela.

###Nela, cada linha representa um bairro, contendo o total de acidentes registrados, o número total de vítimas, o número de vítimas fatais e a taxa de fatalidade (razão entre vítimas fatais e vítimas totais).

###Essa estrutura permite identificar regiões com maior concentração de ocorrências, avaliar a gravidade média dos acidentes e comparar o risco relativo entre diferentes bairros, servindo como base para análises mais profundas e tomada de decisão orientada por dados.

In [0]:
df_bairro = (
    df_pandas
    .groupby("bairro")
    .agg(
        total_acidentes=("bairro", "count"),
        total_vitimas=("vitimas", "sum"),
        total_fatais=("vitimasfatais", "sum")
    )
    .reset_index()
)

# taxa de fatalidade
df_bairro["taxa_fatalidade"] = (
    df_bairro["total_fatais"] / df_bairro["total_vitimas"]
)

# percentual de acidentes por bairro
total_geral_acidentes = df_bairro["total_acidentes"].sum()

df_bairro["percentual_acidentes"] = (
    df_bairro["total_acidentes"] / total_geral_acidentes
) * 100

# ordenar
df_bairro = df_bairro.sort_values(
    by="total_acidentes",
    ascending=False
)




df_bairro.display()

In [0]:
# Convertendo o DataFrame do Pandas para DataFrame do Spark
df_bairro_spark = spark.createDataFrame(df_bairro)

# Grava os dados no formato Delta Lake na camada Gold (Data Lake),
# sobrescrevendo os dados e permitindo evolução do schema
'''df_bairro_spark.write.format("delta") \
  .mode("overwrite") \
  .option("mergeSchema","true") \
  .saveAsTable("/Volumes/workspace/gold/dbs/Sinistros_Transito/Sinistros_Transito_bairro")'''


# Registra a tabela Delta no catálogo a partir dos dados processados
df_bairro_spark.write.format("delta") \
  .mode("overwrite") \
  .option("mergeSchema","true") \
  .saveAsTable("workspace.gold.tb_Sinistros_Transito_bairro")


## Informações por tipo de meio de transporte 

###Queremos entender como os diferentes meios de transporte estão associados aos acidentes. Perguntas como: "Qual tipo de veículo está mais envolvido em acidentes?", "Quais apresentam maior número de vítimas fatais?" e outras análises podem ser respondidas com essa tabela.

###Nela, cada linha representa um tipo de transporte (como carro, moto, pedestre, etc.), contendo o total de ocorrências em que esteve envolvido e o número total de vítimas fatais associadas.

###Essa estrutura permite identificar quais meios estão mais presentes nos acidentes e quais estão relacionados a ocorrências mais graves, servindo como base para análises comparativas e possíveis ações de prevenção.

In [0]:
cols_transportes = ["auto", "moto", "ciclom", "ciclista", "pedestre",
    "onibus", "caminhao", "viatura", "outros"]

data = []

for col in cols_transportes:
    total_acidentes = (df_pandas[col] > 0).sum()
    total_fatais = df_pandas.loc[df_pandas[col] > 0, "vitimasfatais"].sum()
    
    data.append({
        "meio_transporte": col,
        "total_acidentes": total_acidentes,
        "total_fatais": total_fatais,
        "taxa_fatalidade": total_fatais / total_acidentes
        })

df_meio_de_transporte = (
    pd.DataFrame(data)
    .sort_values(by="total_acidentes", ascending=False))

df_meio_de_transporte.display()

In [0]:
# Convertendo o DataFrame do Pandas para DataFrame do Spark
df_transporte_spark = spark.createDataFrame(df_meio_de_transporte)

# Registra a tabela Delta no catálogo a partir dos dados processados
df_transporte_spark.write.format("delta") \
  .mode("overwrite") \
  .option("mergeSchema","true") \
  .saveAsTable("workspace.gold.tb_Sinistros_Transito_transporte")

###Queremos entender como os acidentes se distribuem ao longo do dia. Perguntas como: "Em quais horários ocorrem mais acidentes?" e "Existe algum período mais crítico?" podem ser respondidas com essa tabela.

###Nela, cada linha representa uma hora do dia, contendo o total de acidentes registrados naquele horário.

###Essa estrutura permite identificar padrões temporais, como horários de pico ou períodos de maior risco, servindo como base para análises mais aprofundadas e possíveis ações preventivas.

In [0]:
df_pandas["data"] = pd.to_datetime(df_pandas["data"], errors="coerce")

df_pandas["data_hora"] = df_pandas["data"] + pd.to_timedelta(df_pandas["hora"])

df_hora = (
    df_pandas
    .dropna(subset=["data_hora"])
    .assign(hora_dia=lambda x: x["data_hora"].dt.hour)
    .groupby("hora_dia")
    .agg(total_acidentes=("hora_dia", "count"))
    .reset_index()
    .sort_values(by="hora_dia")
)


df_hora.display()

In [0]:
# Convertendo o DataFrame do Pandas para DataFrame do Spark
df_hora_spark = spark.createDataFrame(df_hora)

# Registra a tabela Delta no catálogo a partir dos dados processados
df_hora_spark.write.format("delta") \
  .mode("overwrite") \
  .option("mergeSchema","true") \
  .saveAsTable("workspace.gold.tb_Sinistros_Transito_hora")

# Aqui, podemos observar que nossos dados são apenas dados parciais entre a madrugada e meio dia. Logo, nossa análise será também limitada.

###Agora, queremos entender como os acidentes evoluem ao longo dos meses. Perguntas como: "Qual mês teve mais acidentes?", "Em quais períodos há mais mortes?" e "A taxa de fatalidade varia ao longo do ano?" podem ser respondidas com essa tabela.

###Nela, cada linha representa um mês, contendo o total de acidentes, o número de vítimas fatais e a taxa de fatalidade (razão entre vítimas fatais e vítimas totais).

###Essa estrutura permite identificar padrões sazonais, comparar períodos mais críticos e avaliar a gravidade dos acidentes ao longo do tempo.

In [0]:
df_mes = (
    df_pandas
    .groupby("mes")
    .agg(
        total_acidentes=("mes", "count"),
        total_vitimas=("vitimas", "sum"),
        total_fatais=("vitimasfatais", "sum")
    )
    .reset_index()
)

# taxa de fatalidade
df_mes["taxa_fatalidade"] = df_mes["total_fatais"] / df_mes["total_vitimas"]

# ordenar por mês
df_mes = df_mes.sort_values(by="mes")

df_mes.display()

In [0]:
# Convertendo o DataFrame do Pandas para DataFrame do Spark
df_mes_spark = spark.createDataFrame(df_mes)

# Registra a tabela Delta no catálogo a partir dos dados processados
df_mes_spark.write.format("delta") \
  .mode("overwrite") \
  .option("mergeSchema","true") \
  .saveAsTable("workspace.gold.tb_Sinistros_Transito_mes")